# GEDI Canopy Cover Pipeline

This notebook processes NASA GEDI Level 2B HDF5 files stored in S3 through four sequential phases:

| Phase | Description |
|---|---|
| **Phase 1** | High-performance batch extraction of GEDI shots from S3 |
| **Phase 1a** | Year parsing from GEDI filenames |
| **Phase 1b** | Harmonization to the SMAP 9 km grid |
| **Phase 1c** | Spatial join to assign shots to study-area jurisdictions |

**Source:** `s3://central-virginia-tree-canopy-project/GEDI/GEDI02_B/002/`  
**Final Summary output:** `s3://central-virginia-tree-canopy-project/gedi02B-county-summary/virginia_gedi02B_county_summary.csv`
**Final Detailed output:** `s3://central-virginia-tree-canopy-project/gedi-county-detailed/virginia_gedi02B_county_canopy_height.csv`

## Cell 1 — Install Dependencies

In [1]:
import subprocess, sys

# Pin the entire boto stack atomically — all four packages must be compatible
boto_stack = [
    "botocore==1.43.0",
    "boto3==1.43.0",
    "s3fs==2026.6.0",
]

# Install boto stack with --no-deps to prevent pip from overriding the pins
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet",
                       "--force-reinstall"] + boto_stack)

# Install all other dependencies normally
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet",
                       "h5py", "geopandas", "xarray", "netCDF4", "pyarrow"])

print("Boto Stack and all dependencies installed.")

All dependencies installed.


## Cell 2 — Imports

In [2]:
import os
import time
import io
import re
import urllib.request

from concurrent.futures import ThreadPoolExecutor, as_completed

import statistics

import boto3
import s3fs
import h5py
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr

print("Imports successful.")

Imports successful.


## Cell 3 — Configuration

When necessary, edit this cell to change S3 paths, bounding box, grid resolution, or target jurisdictions.

In [4]:
# ── S3 settings ───────────────────────────────────────────────────────────────
S3_BUCKET             = "central-virginia-tree-canopy-project"
S3_PREFIX             = "GEDI/GEDI02_B/002/"
GEDI02B_COUNTY_S3_PREFIX = "gedi02B-county-summary/"
GEDI02B_DETAILED_COUNTY_S3_PREFIX = "gedi02B-county-detailed/"

# ── Local output directory ────────────────────────────────────────────────────
OUTPUT_DIR        = "/home/ec2-user/SageMaker/gedi_output"
OUTPUT_PARQUET    = os.path.join(OUTPUT_DIR, "virginia_gedi02B_canopy-cover_multiyear.parquet")
OUTPUT_NETCDF     = os.path.join(OUTPUT_DIR, "virginia_gedi02B_canopy-cover_grid.nc")
OUTPUT_COUNTY_CSV = os.path.join(OUTPUT_DIR, "virginia_gedi02B_county-cover_summary.csv")
OUTPUT_DETAILED_COUNTY_CSV = os.path.join(OUTPUT_DIR, "virginia_gedi02B_county_canopy_cover.csv")
OUTPUT_DETAILED_COUNTY_CANOPY_COVER_CSV = os.path.join(OUTPUT_DIR, "virginia_gedi02B_county_canopy_cover.csv")

# ── Spatial bounds (Virginia 8-jurisdiction study area) ───────────────────────
MIN_LON, MIN_LAT, MAX_LON, MAX_LAT = -79.1721, 37.3296, -77.6873, 38.4755

# ── SMAP grid resolution (~9 km) ─────────────────────────────────────────────
GRID_RES = 0.081

# ── Target jurisdictions (Name, State FIPS, County/Place FIPS, Type) ─────────
TARGET_JURISDICTIONS = [
    ("Albemarle",       "51", "003",   "county"),
    ("Augusta",         "51", "015",   "county"),
    ("Buckingham",      "51", "029",   "county"),
    ("Charlottesville", "51", "14968", "place"),
    ("Fluvanna",        "51", "065",   "county"),
    ("Greene",          "51", "079",   "county"),
    ("Louisa",          "51", "109",   "county"),
    ("Nelson",          "51", "125",   "county"),
    ("Orange",          "51", "137",   "county"),
    ("Rockingham",      "51", "165",   "county"),
]

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Configuration loaded.")
print(f"  GEDI02_B source  : s3://{S3_BUCKET}/{S3_PREFIX}")
print(f"  Output dir   : {OUTPUT_DIR}")
print(f"  S3 output    : s3://{S3_BUCKET}/{GEDI02B_COUNTY_S3_PREFIX}")

Configuration loaded.
  GEDI02_B source  : s3://central-virginia-tree-canopy-project/GEDI/GEDI02_B/002/
  Output dir   : /home/ec2-user/SageMaker/gedi_output
  S3 output    : s3://central-virginia-tree-canopy-project/gedi02B-county-summary/


## Cell 4 — Helper Functions

In [5]:
def parse_year_from_filename(filename: str) -> int:
    """Extract the year from standard GEDI02_B filename (e.g., GEDI02_B_2022143...)."""
    year_match = re.search(r'GEDI02_B_(\d{4})', filename)
    if year_match:
        return int(year_match.group(1))
    return None

def fetch_boundary(name: str, state_fips: str, geo_id: str, geo_type: str) -> gpd.GeoDataFrame:
    """Fetch boundary GeoJSON directly from US Census TIGERweb API."""
    if geo_type == "place":
        url = (
            "https://tigerweb.geo.census.gov/arcgis/rest/services/TIGERweb/"
            "Places_CouSub_ConCity_SubMCD/MapServer/4/query"
            f"?where=STATE='{state_fips}'+AND+PLACE='{geo_id}'"
            "&outFields=NAME,STATE,PLACE&f=geojson&outSR=4326"
        )
    else:
        url = (
            "https://tigerweb.geo.census.gov/arcgis/rest/services/TIGERweb/"
            "State_County/MapServer/11/query"
            f"?where=STATE='{state_fips}'+AND+COUNTY='{geo_id}'"
            "&outFields=NAME,STATE,COUNTY&f=geojson&outSR=4326"
        )
    
    print(f"Fetching boundary for {name}...")
    try:
        with urllib.request.urlopen(url, timeout=20) as r:
            gdf = gpd.read_file(r)
        if gdf.empty:
            raise ValueError(f"No boundary found for {name}")
        gdf = gdf.set_crs("EPSG:4326")
        gdf['jurisdiction'] = name
        return gdf
    except Exception as e:
        print(f"Failed to fetch boundary for {name}: {e}")
        return gpd.GeoDataFrame()

print("Helper functions defined.")

Helper functions defined.


## Cell 5 — Phase 1 & 1a: Discover GEDI02_B Files in S3

In [6]:
print(f"Scanning s3://{S3_BUCKET}/{S3_PREFIX} for GEDI02_B HDF5 files...")
s3 = boto3.client("s3")
paginator = s3.get_paginator("list_objects_v2")

h5_keys = []
for page in paginator.paginate(Bucket=S3_BUCKET, Prefix=S3_PREFIX):
    for obj in page.get("Contents", []):
        if obj["Key"].endswith(".h5") and "GEDI02_B" in obj["Key"]:
            h5_keys.append(obj["Key"])
            
print(f"Found {len(h5_keys)} GEDI02_B files to process.")
if h5_keys:
    print(f"  First : {h5_keys[0]}")
    print(f"  Last  : {h5_keys[-1]}")

Scanning s3://central-virginia-tree-canopy-project/GEDI/GEDI02_B/002/ for GEDI02_B HDF5 files...
Found 394 GEDI02_B files to process.
  First : GEDI/GEDI02_B/002/GEDI02_B_2019113174925_O02048_03_T02621_02_003_01_V002.h5
  Last  : GEDI/GEDI02_B/002/GEDI02_B_2025189013841_O37209_02_T08829_02_004_01_V002.h5


## Cell 6 — Phase 1 & 1a: Batch Extraction with Spatial Masking and Quality Filtering

In [ ]:
# Batch extraction helper function

def extract_gedi_dataframes(h5_keys, max_workers=16):
    """
    Concurrently download and process GEDI .h5 granules from S3, returning
    a combined list of per-beam DataFrames that passed spatial/quality filtering.

    Parameters
    ----------
    h5_keys : list[str]
        S3 object keys for the .h5 granules to process.
    max_workers : int, optional
        Number of concurrent threads to use for downloading/processing files
        (default: 16). Network I/O bound, so pushing this higher is usually
        safe until you hit S3 throttling or bandwidth limits.

    Returns
    -------
    list[pd.DataFrame]
        List of per-beam DataFrames, one per beam that had valid data after
        spatial masking and quality filtering.
    """
    all_dfs = []
    cell_start_time = time.time()
    completed = 0

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(process_one_file, key): key for key in h5_keys}
        for future in as_completed(futures):
            key = futures[future]
            try:
                file_dfs = future.result()
                all_dfs.extend(file_dfs)
            except Exception as e:
                print(f"Unhandled error processing {os.path.basename(key)}: {e}")
            completed += 1
            if completed % 10 == 0:
                elapsed = time.time() - cell_start_time
                print(f"Processed {completed}/{len(h5_keys)} files... "
                      f"({elapsed:.2f}s elapsed total)")

    total_elapsed = time.time() - cell_start_time
    minutes, seconds = divmod(total_elapsed, 60)
    print(f"\nExtraction complete. Beams with valid data: {len(all_dfs)}")
    print(f"🚀 Total execution time: {int(minutes)}m {seconds:.2f}s")

    return all_dfs

print("Batch extraction helper function defined.")

In [7]:
fs = s3fs.S3FileSystem(anon=False)

all_dfs = extract_gedi_dataframes(h5_keys, max_workers=9)  #Max Workers can be set to 16 or 24 

# # This is a long running processing....let's time it
# cell_start_time = time.time()

# print("\n--- Starting Phase 1: Point Extraction (GEDI02_B) ---")
# for i, key in enumerate(h5_keys):
#     file_start_time = time.time()
#     file_name = os.path.basename(key)
#     year = parse_year_from_filename(file_name)
    
#     if not year:
#         continue
        
#     try:
#         s3_path = f"s3://{S3_BUCKET}/{key}"
#         #print(f"DEBUG: Current s3_path: {s3_path}")
#         with fs.open(s3_path, "rb") as f:
#             raw_bytes = f.read()
            
#         with h5py.File(io.BytesIO(raw_bytes), 'r') as hf:
#             beams = [k for k in hf.keys() if k.startswith('BEAM')]
            
#             for beam in beams:
#                 # GEDI02_B houses geographic coordinates within a sub-group
#                 if f"{beam}/geolocation/lat_lowestmode" not in hf:
#                     continue
                    
#                 lats = hf[f"{beam}/geolocation/lat_lowestmode"][:]
#                 lons = hf[f"{beam}/geolocation/lon_lowestmode"][:]
                
#                 spatial_mask = (lons >= MIN_LON) & (lons <= MAX_LON) & (lats >= MIN_LAT) & (lats <= MAX_LAT)
#                 if not np.any(spatial_mask):
#                     continue
                    
#                 # Extract L2B specific metrics and apply mask
#                 quality = hf[f"{beam}/l2b_quality_flag"][:][spatial_mask]
#                 sensitivity = hf[f"{beam}/sensitivity"][:][spatial_mask]
                
#                 # Target metric: Total Canopy Cover (values scale from 0.0 to 1.0)
#                 #canopy_cover = hf[f"{beam}/canopy_cover"][:][spatial_mask]
#                 #The key, canopy_cover, is non-existent in the GEDI HDF5 schema
#                 #Actual Error Message
#                 #--- Starting Phase 1: Point Extraction (GEDI02_B) ---
#                 #Current s3_path: s3://central-virginia-tree-canopy-project/GEDI/GEDI02_B/002/GEDI02_B_2019113174925_O02048_03_T02621_02_003_01_V002.h5
#                 #Error reading GEDI02_B_2019113174925_O02048_03_T02621_02_003_01_V002.h5: "Unable to synchronously open object (object 'canopy_cover' doesn't exist)"

#                 canopy_cover = hf[f"{beam}/cover"][:][spatial_mask]
                
#                 beam_df = pd.DataFrame({
#                     'longitude': lons[spatial_mask],
#                     'latitude': lats[spatial_mask],
#                     'l2b_quality_flag': quality,
#                     'sensitivity': sensitivity,
#                     'canopy_cover': canopy_cover,
#                     'year': year,
#                     'file_source': file_name,
#                     'beam': beam
#                 })
                
#                 # Quality filtering tailored for Canopy Cover
#                 valid_df = beam_df[
#                     (beam_df['l2b_quality_flag'] == 1) & 
#                     (beam_df['sensitivity'] > 0.9) & 
#                     (beam_df['canopy_cover'] >= 0.0) & 
#                     (beam_df['canopy_cover'] <= 1.0)
#                 ]
                
#                 if not valid_df.empty:
#                     all_dfs.append(valid_df)
                    
#     except Exception as e:
#         print(f"Error reading {file_name}: {e}")
        
#     if (i + 1) % 10 == 0:
#         file_elapsed = time.time() - file_start_time
#         print(f"Processed {i + 1}/{len(h5_keys)} files... in {file_elapsed:.2f} seconds.")
#         #print(f"Processed {i + 1}/{len(h5_keys)} files...")

# #Calculate and display total run time
# total_elapsed = time.time() - cell_start_time
# minutes, seconds = divmod(total_elapsed, 60)

# print(f"\nExtraction complete. Beams with valid data: {len(all_dfs)}")
# print(f"🚀 Total execution time: {int(minutes)}m {seconds:.2f}s")


--- Starting Phase 1: Point Extraction (GEDI02_B) ---
Processed 10/394 files... in 8.33 seconds.
Processed 20/394 files... in 9.60 seconds.
Processed 30/394 files... in 17.34 seconds.
Processed 40/394 files... in 16.30 seconds.
Processed 50/394 files... in 7.99 seconds.
Processed 60/394 files... in 13.72 seconds.
Processed 70/394 files... in 9.82 seconds.
Processed 80/394 files... in 15.04 seconds.
Processed 90/394 files... in 10.77 seconds.
Processed 100/394 files... in 15.98 seconds.
Processed 110/394 files... in 16.33 seconds.
Processed 120/394 files... in 14.95 seconds.
Processed 130/394 files... in 8.60 seconds.
Processed 140/394 files... in 10.36 seconds.
Processed 150/394 files... in 14.33 seconds.
Processed 160/394 files... in 12.64 seconds.
Processed 170/394 files... in 13.54 seconds.
Processed 180/394 files... in 8.26 seconds.
Processed 190/394 files... in 14.15 seconds.
Processed 200/394 files... in 6.15 seconds.
Processed 210/394 files... in 8.86 seconds.
Processed 220/394

## Cell 7 — Save Extracted Points to Parquet

In [8]:
if not all_dfs:
    raise RuntimeError("No valid GEDI02_B shots found within the bounding box. Check S3 prefix and bbox settings.")
        
df_gedi = pd.concat(all_dfs, ignore_index=True)
df_gedi.to_parquet(OUTPUT_PARQUET, index=False)

print(f"Total valid GEDI shots saved : {len(df_gedi):,}")
print(f"Years covered                : {sorted(df_gedi['year'].unique())}")
print(f"Parquet file                 : {OUTPUT_PARQUET}")
df_gedi.head()

Total valid GEDI shots saved : 2,074,219
Years covered                : [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Parquet file                 : /home/ec2-user/SageMaker/gedi_output/virginia_gedi02B_canopy-cover_multiyear.parquet


,longitude,latitude,l2b_quality_flag,sensitivity,canopy_cover,year,file_source,beam
0,-79.171967,38.044788,1,0.900244,0.013730,2019,GEDI02_B_2019113174925_O02048_03_T02621_02_003...,BEAM0000
1,-79.171469,38.044455,1,0.916953,0.010623,2019,GEDI02_B_2019113174925_O02048_03_T02621_02_003...,BEAM0000
2,-79.170971,38.044123,1,0.918692,0.002983,2019,GEDI02_B_2019113174925_O02048_03_T02621_02_003...,BEAM0000
3,-79.170475,38.043790,1,0.912793,0.311442,2019,GEDI02_B_2019113174925_O02048_03_T02621_02_003...,BEAM0000
4,-79.169977,38.043457,1,0.907017,0.350270,2019,GEDI02_B_2019113174925_O02048_03_T02621_02_003...,BEAM0000


In [9]:
#Save to CSV to support Canopy Cover Change
df_gedi.to_csv('/home/ec2-user/SageMaker/gedi_output/virginia_gedi_canopy_cover.csv', index=False)
print(f"Saved locally to : /home/ec2-user/SageMaker/gedi_output/virginia_gedi_canopy_cover.csv")

Saved locally to : /home/ec2-user/SageMaker/gedi_output/virginia_gedi_canopy_cover.csv


## Cell 8 — Phase 1b: Harmonize to the SMAP 9 km Grid

In [10]:
print("\n--- Starting Phase 1b: Grid Harmonization ---")
lon_bins = np.arange(MIN_LON, MAX_LON + GRID_RES, GRID_RES)
lat_bins = np.arange(MIN_LAT, MAX_LAT + GRID_RES, GRID_RES)

df_gedi['lon_grid'] = pd.cut(df_gedi['longitude'], bins=lon_bins, labels=lon_bins[:-1]).astype(float)
df_gedi['lat_grid'] = pd.cut(df_gedi['latitude'], bins=lat_bins, labels=lat_bins[:-1]).astype(float)

gedi_grid = df_gedi.groupby(['year', 'lat_grid', 'lon_grid'])['canopy_cover'].mean().reset_index()

ds_gedi = gedi_grid.set_index(['year', 'lat_grid', 'lon_grid']).to_xarray()
ds_gedi.to_netcdf(OUTPUT_NETCDF)

print(f"Grid02_B cells produced : {len(gedi_grid):,}")
print(f"NetCDF saved to     : {OUTPUT_NETCDF}")
gedi_grid.head()


--- Starting Phase 1b: Grid Harmonization ---
Grid02_B cells produced : 1,815
NetCDF saved to     : /home/ec2-user/SageMaker/gedi_output/virginia_gedi02B_canopy-cover_grid.nc


,year,lat_grid,lon_grid,canopy_cover
0,2019,37.3296,-79.1721,0.367706
1,2019,37.3296,-79.0911,0.394191
2,2019,37.3296,-79.0101,0.359417
3,2019,37.3296,-78.9291,0.312805
4,2019,37.3296,-78.8481,0.349452


## Cell 9 — Phase 1c: Fetch Jurisdiction Boundaries from US Census TIGERweb

In [11]:
print("\n--- Starting Phase 2: Zonal Statistics Summary ---")
gdf_boundaries = []

for name, state, fips, g_type in TARGET_JURISDICTIONS:
    try:
        bound_df = fetch_boundary(name, state, fips, g_type)
        gdf_boundaries.append(bound_df)
    except Exception as e:
        print(f"  Failed to fetch boundary for {name}: {e}")
        
if not gdf_boundaries:
    raise RuntimeError("No jurisdiction boundaries fetched. Cannot perform spatial join.")

filtered_counties = pd.concat(gdf_boundaries, ignore_index=True)
print(f"\nBoundaries fetched: {len(filtered_counties)} jurisdiction(s)")
filtered_counties[['jurisdiction', 'geometry']].head(10)


--- Starting Phase 2: Zonal Statistics Summary ---
Fetching boundary for Albemarle...
Fetching boundary for Augusta...
Fetching boundary for Buckingham...
Fetching boundary for Charlottesville...
Fetching boundary for Fluvanna...
Fetching boundary for Greene...
Fetching boundary for Louisa...
Fetching boundary for Nelson...
Fetching boundary for Rockingham...

Boundaries fetched: 9 jurisdiction(s)


,jurisdiction,geometry
0,Albemarle,"POLYGON ((-78.64831 37.73272, -78.64824 37.732..."
1,Augusta,"POLYGON ((-78.93069 38.31342, -78.93242 38.314..."
2,Buckingham,"POLYGON ((-78.24812 37.64089, -78.248 37.64158..."
3,Charlottesville,"POLYGON ((-78.52368 38.02233, -78.52371 38.022..."
4,Fluvanna,"POLYGON ((-78.15905 37.74933, -78.15889 37.749..."
5,Greene,"POLYGON ((-78.3587 38.29019, -78.35879 38.2904..."
6,Louisa,"POLYGON ((-78.30676 38.00647, -78.30687 38.006..."
7,Nelson,"POLYGON ((-78.90797 37.97698, -78.90798 37.976..."
8,Rockingham,"POLYGON ((-79.09295 38.66441, -79.09295 38.664..."


## Cell 10 — Phase 1c: Spatial Join and County-Level Aggregation

In [12]:
gdf_regions = pd.concat(gdf_boundaries, ignore_index=True)

gdf_points = gpd.GeoDataFrame(
    df_gedi, 
    geometry=gpd.points_from_xy(df_gedi.longitude, df_gedi.latitude), 
    crs="EPSG:4326"
)
    
print("Performing spatial join...")
# Spatial Join points to target boundaries
gedi02B_with_county = gpd.sjoin(
    gdf_points, 
    gdf_regions, 
    how="inner", 
    predicate="within"
)

gedi02B_county_summary = (
    gedi02B_with_county
    .groupby(['jurisdiction', 'year']).agg(
    mean_canopy_cover=('canopy_cover', 'mean'),
    total_valid_shots=('canopy_cover', 'count')
    ).reset_index()
)

print(f"Spatial join complete. Rows in summary: {len(gedi02B_county_summary)}")
gedi02B_county_summary

Performing spatial join...
Spatial join complete. Rows in summary: 61


,jurisdiction,year,mean_canopy_cover,total_valid_shots
0,Albemarle,2019,0.529159,53356
1,Albemarle,2020,0.540116,14835
2,Albemarle,2021,0.424221,54570
3,Albemarle,2022,0.459664,47618
4,Albemarle,2023,0.207803,11110
...,...,...,...,...
56,Rockingham,2021,0.368383,23337
57,Rockingham,2022,0.293317,29597
58,Rockingham,2023,0.281510,6661
59,Rockingham,2024,0.293816,18177


In [13]:
# Load dataframe with canopy cover data points
df_gedi_canopy_cover = pd.read_csv('/home/ec2-user/SageMaker/gedi_output/virginia_gedi_canopy_cover.csv')

gdf_gedi_canopy_cover = gpd.GeoDataFrame(
     df_gedi_canopy_cover,
     geometry=gpd.points_from_xy(df_gedi_canopy_cover.longitude, df_gedi_canopy_cover.latitude),
     crs="EPSG:4326"
 )

# The Fix for the Value Error

# Drop any stale index columns from a previous join
gdf_gedi_canopy_cover = gdf_gedi_canopy_cover.drop(
    columns=[c for c in ["index_right", "index_left"] if c in gdf_gedi_canopy_cover.columns]
)
filtered_counties = filtered_counties.drop(
    columns=[c for c in ["index_right", "index_left"] if c in filtered_counties.columns]
)

print("Performing spatial join...")
gedi_with_county_canopy_cover = gpd.sjoin(
    gdf_gedi_canopy_cover,
    filtered_counties[['jurisdiction', 'geometry']],
    how='inner',
    predicate='within'
)

# gedi_county_summary = (
#     gedi_with_county
#     .groupby(['year', 'jurisdiction'])['rh98']
#     .mean()
#     .reset_index()
#     .rename(columns={'rh98': 'canopy_height_mean_m'})
# )

print(f"Spatial join complete. Rows in summary: {len(gedi_with_county_canopy_cover)}")
gedi_with_county_canopy_cover.head(20)

Performing spatial join...
Spatial join complete. Rows in summary: 1147795


,longitude,latitude,l2b_quality_flag,sensitivity,canopy_cover,year,file_source,beam,geometry,index_right,jurisdiction
0,-79.171967,38.044788,1,0.900244,0.013730,2019,GEDI02_B_2019113174925_O02048_03_T02621_02_003...,BEAM0000,POINT (-79.17197 38.04479),1,Augusta
1,-79.171469,38.044455,1,0.916953,0.010623,2019,GEDI02_B_2019113174925_O02048_03_T02621_02_003...,BEAM0000,POINT (-79.17147 38.04446),1,Augusta
2,-79.170971,38.044123,1,0.918692,0.002983,2019,GEDI02_B_2019113174925_O02048_03_T02621_02_003...,BEAM0000,POINT (-79.17097 38.04412),1,Augusta
3,-79.170475,38.043790,1,0.912793,0.311442,2019,GEDI02_B_2019113174925_O02048_03_T02621_02_003...,BEAM0000,POINT (-79.17047 38.04379),1,Augusta
4,-79.169977,38.043457,1,0.907017,0.350270,2019,GEDI02_B_2019113174925_O02048_03_T02621_02_003...,BEAM0000,POINT (-79.16998 38.04346),1,Augusta
5,-79.169478,38.043124,1,0.961559,0.355211,2019,GEDI02_B_2019113174925_O02048_03_T02621_02_003...,BEAM0000,POINT (-79.16948 38.04312),1,Augusta
6,-79.168979,38.042791,1,0.961136,0.230824,2019,GEDI02_B_2019113174925_O02048_03_T02621_02_003...,BEAM0000,POINT (-79.16898 38.04279),1,Augusta
7,-79.168481,38.042459,1,0.915506,0.214787,2019,GEDI02_B_2019113174925_O02048_03_T02621_02_003...,BEAM0000,POINT (-79.16848 38.04246),1,Augusta
8,-79.167985,38.042126,1,0.909027,0.219677,2019,GEDI02_B_2019113174925_O02048_03_T02621_02_003...,BEAM0000,POINT (-79.16798 38.04213),1,Augusta
9,-79.167488,38.041793,1,0.921234,0.056018,2019,GEDI02_B_2019113174925_O02048_03_T02621_02_003...,BEAM0000,POINT (-79.16749 38.04179),1,Augusta


In [14]:
#Save to CSV to support County Canopy Cover Change
gedi_with_county_canopy_cover.to_csv(OUTPUT_DETAILED_COUNTY_CANOPY_COVER_CSV, index=False)
print(f"Saved locally to : {OUTPUT_DETAILED_COUNTY_CANOPY_COVER_CSV}")
#gedi_with_county_canopy_cover.to_csv('/home/ec2-user/SageMaker/gedi_output/virginia_gedi_county_canopy_cover.csv', index=False)
#print(f"Saved locally to : /home/ec2-user/SageMaker/gedi_output/virginia_gedi_county_canopy_cover.csv")

Saved locally to : /home/ec2-user/SageMaker/gedi_output/virginia_gedi_county_canopy_cover.csv


## Cell 11 — Save County Summary Locally and Upload to S3

In [15]:
# Save locally

gedi02B_county_summary.to_csv(OUTPUT_COUNTY_CSV, index=False)
print(f"Saved locally to : {OUTPUT_COUNTY_CSV}")

# Upload to S3 for downstream pipeline consumption
county_csv_filename = os.path.basename(OUTPUT_COUNTY_CSV)
s3_county_key       = GEDI02B_COUNTY_S3_PREFIX + county_csv_filename
s3_client           = boto3.client("s3")

s3_client.upload_file(
    OUTPUT_COUNTY_CSV,
    S3_BUCKET,
    s3_county_key,
    ExtraArgs={"ContentType": "text/csv"}
)

s3_county_uri = f"s3://{S3_BUCKET}/{s3_county_key}"
print(f"Uploaded to S3   : {s3_county_uri}")
print("\nAll phases completed successfully!")

Saved locally to : /home/ec2-user/SageMaker/gedi_output/virginia_gedi02B_county-cover_summary.csv
Uploaded to S3   : s3://central-virginia-tree-canopy-project/gedi02B-county-summary/virginia_gedi02B_county-cover_summary.csv

All phases completed successfully!


## Cell 12 — Save Detailed County locally and Upload to S3

In [ ]:
# Save detailed county output to S3
gedi_with_county_canopy_cover.to_csv(OUTPUT_DETAILED_COUNTY_CANOPY_COVER_CSV, index=False)
print(f"Saved locally to : {OUTPUT_DETAILED_COUNTY_CANOPY_COVER_CSV}")

# Upload to S3 for downstream pipeline consumption
county_csv_filename = os.path.basename(OUTPUT_DETAILED_COUNTY_CANOPY_COVER_CSV)
s3_county_key       = GEDI_DETAILED_COUNTY_S3_PREFIX + county_csv_filename
s3_client           = boto3.client("s3")

s3_client.upload_file(
    OUTPUT_DETAILED_COUNTY_CANOPY_COVER_CSV,
    S3_BUCKET,
    s3_county_key,
    ExtraArgs={"ContentType": "text/csv"}
)

s3_county_uri = f"s3://{S3_BUCKET}/{s3_county_key}"
print(f"Uploaded to S3   : {s3_county_uri}")
print("\nAll detailed phases completed successfully!")